# Example: Super-fast mode (no Grad-Shafranov solves)

## Import packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import pickle

## Build the machine

In [ ]:
# build machine
from freegsnke import build_machine
tokamak = build_machine.tokamak(
    active_coils_path="../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path="../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path="../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path="../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

### Instantiate an equilibrium

In [ ]:
from freegsnke import equilibrium_update

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1, Rmax=2.0,   # Radial range
    Zmin=-2.2, Zmax=2.2,  # Vertical range
    nx=65,                # Number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=129,               # Number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
    # psi=plasma_psi
)  

### Instatiate a profile object

In [ ]:
from freegsnke.jtor_update import ConstrainPaxisIp

profiles = ConstrainPaxisIp(
    eq=eq,        # equilibrium object
    paxis=8.1e3,  # profile object
    Ip=6.2e5,     # plasma current
    fvac=0.5,     # fvac = rB_{tor}
    alpha_m=1.8,  # profile function parameter
    alpha_n=1.2   # profile function parameter
)

### Set coil currents
Here we set coil currents that create a diverted plasma (as seen in prior notebooks). 

In [ ]:
with open('data/simple_diverted_currents_PaxisIp.pk', 'rb') as f:
    current_values = pickle.load(f)

for key in current_values.keys():
    eq.tokamak.set_coil_current(coil_label=key, current_value=current_values[key])

### Instatiate the solver


In [ ]:
from freegsnke import GSstaticsolver
GSStaticSolver = GSstaticsolver.NKGSsolver(eq)    

### Call forward solver to find equilibrium 

In [ ]:
GSStaticSolver.solve(
    eq=eq, 
    profiles=profiles, 
    constrain=None, 
    target_relative_tolerance=1e-8,
    verbose=0
)

### Plot the initial equilibrium 

In [ ]:
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

### Define descriptors

Descriptors are quantaties of interest that will be approximated via a linearisation from the initial equilibrium. This means it will be possible to find key time-dependant quantities of interest e.g. `z` without solving the computationally expensive Grad-Shafranov equation.


In [ ]:
# choose the x-point in XPT_BOX for calculating
# certain plasma descriptors 
XPT_BOX = [[0.33, -0.88], [0.95, -1.38]]

def plasma_descriptors(eq):
    xpt_mask = (
        (eq.xpt[:, 0] >= XPT_BOX[0][0])
        & (eq.xpt[:, 0] <= XPT_BOX[1][0])
        & (eq.xpt[:, 1] <= XPT_BOX[0][1])
        & (eq.xpt[:, 1] >= XPT_BOX[1][1])
    )

    xpts = eq.xpt[xpt_mask, 0:2].squeeze()
    if xpts.ndim > 1 and xpts.shape[0] > 1:
        opt = eq.opt[0, 0:2]
        dists = np.linalg.norm(xpts - opt, axis=1)
        idx = np.argmin(dists)  # index of closest point
        rx, zx = xpts[idx, :]
    else:
        rx, zx = xpts

    # z, Rx, Zx
    return np.array(
        [eq.Zcurrent(), rx, zx]
    )

### Time evolution

We are now ready to solve the forward time-evolutive problem. This will follow the same setup as example 05.

In [ ]:
from freegsnke import nonlinear_solve

stepping = nonlinear_solve.nl_solver(
    eq=eq, 
    profiles=profiles, 
    GSStaticSolver=GSStaticSolver,
    full_timestep=5e-4, 
    plasma_resistivity=1e-6,
    max_mode_frequency=10**2.5,
    plasma_descriptor_function=plasma_descriptors
)

#### Set active coil voltages

Again, like example 05, we will evolve the plasma with no control or external drive.

In [ ]:
voltages = (stepping.vessel_currents_vec*stepping.evol_metal_curr.coil_resist)[:stepping.evol_metal_curr.n_active_coils] 

To start, the solver is prepared by setting the initial conditions.

In [ ]:
stepping.initialize_from_ICs(eq, profiles)

#### Set time steps
Now we set the total number of time steps we want to simulate

In [ ]:
# number of time steps to simulate
max_count = 90

# initialising some variables for iteration and logging
counter = 0
t = 0

#### Set time-dependent active coil voltages, profile parameters, and plasma resistivity


In [ ]:
voltages = (stepping.vessel_currents_vec*stepping.evol_metal_curr.coil_resist)[:stepping.evol_metal_curr.n_active_coils] 

# we define some time-dependent plasma current density profile parameters here
alpha_m = np.tile(profiles.alpha_m, max_count+1)
alpha_m -= (0.1 * np.sin(0.05 * np.pi * np.arange(max_count+1))) # we add some perturbation

alpha_n = np.tile(profiles.alpha_n, max_count+1)
alpha_n += (0.1 * np.sin(0.1 * np.pi * np.arange(max_count+1))) # we add some perturbation

paxis = np.tile(profiles.paxis, max_count+1)
paxis += (0.1 * np.sin(0.01 * np.pi * np.arange(max_count+1))) # we add some perturbation

#### Call the solver (no GS)
Finally, we call the time-evolutive solver with `stepping.nlstepper()` sequentially until we reach the preset end time.

TODO

In [ ]:
# initialise the solver with the initial equilibrium/profiles
stepping.initialize_from_ICs(eq, profiles)

# needs to be done here because the linearisation is built in the line above
plasma_descriptors_vector = stepping.new_plasma_descriptors(
    stepping.initial_currents_plasma_descriptor, 
    stepping.initial_profiles_plasma_descriptor
)
history_estimated_plasma_descriptors = [plasma_descriptors_vector]

history_times = [t]

history_currents = [stepping.currents_vec]
history_equilibria = [stepping.eq1.create_auxiliary_equilibrium()]

In [ ]:
# loop over time steps
while counter<max_count:
    print(f'Step: {counter}/{max_count-1}')
    print(f'--- t = {t:.2e}')


    # carry out the time step (profile parameters and resistivity are constant at each time step)
    stepping.nlstepper(
        active_voltage_vec=voltages,   # same voltages used at each time step
        linear_only=True,
        no_GS=True,
        verbose=False,
        max_solving_iterations=50,
        profiles_parameters = {
            "alpha_m": alpha_m[counter],
            "alpha_n": alpha_n[counter],
            "paxis": paxis[counter],
        },
    )

    plasma_descriptors_vector = stepping.new_plasma_descriptors(
        new_currents=stepping.currents_vec, 
        new_profiles=np.array([
            stepping.profiles1.alpha_m,
            stepping.profiles1.alpha_n,
            getattr(stepping.profiles1, stepping.profiles_param)
        ])
    )

    # store information on the time step
    t += stepping.dt_step
    history_times.append(t)
    counter += 1

    # store time-advanced equilibrium, currents, and profiles (+ other quantites of interest)
    history_currents.append(stepping.currents_vec)
    history_equilibria.append(stepping.eq1.create_auxiliary_equilibrium())
    history_estimated_plasma_descriptors.append(plasma_descriptors_vector)

# transform lists to arrays
history_currents = np.array(history_currents)
history_times = np.array(history_times)
history_estimated_plasma_descriptors = np.array(history_estimated_plasma_descriptors)


#### Call the solver (linear)

TODO

In [ ]:
# reset the solver object by resetting the initial conditions
stepping.initialize_from_ICs(eq, profiles)

# initialising some variables for iteration and logging
counter = 0
t = 0

# store the true descriptors this time, not an approximation
history_plasma_descriptors_linear = [plasma_descriptors(stepping.eq1)]

history_times_linear = [t]
history_currents_linear = [stepping.currents_vec]
history_equilibria_linear = [stepping.eq1.create_auxiliary_equilibrium()]

# loop over the time steps
while counter<max_count:

    print(f'Step: {counter}/{max_count-1}')
    print(f'--- t = {t:.2e}')
    
    # carry out the time step
    stepping.nlstepper(
        active_voltage_vec=voltages, 
        linear_only=True,
        no_GS=False,
        verbose=False,
        max_solving_iterations=50,
        profiles_parameters = {
            "alpha_m": alpha_m[counter],
            "alpha_n": alpha_n[counter],
            "paxis": paxis[counter],
        },
    )

    # store information on the time step
    t += stepping.dt_step
    history_times_linear.append(t)
    counter += 1
    
    # store time-advanced equilibrium, currents, and profiles (+ other quantites of interest)
    history_currents_linear.append(stepping.currents_vec)
    history_equilibria_linear.append(stepping.eq1.create_auxiliary_equilibrium())
    history_plasma_descriptors_linear.append(plasma_descriptors(stepping.eq1))
    

# transform lists to arrays
history_currents_linear = np.array(history_currents_linear)
history_times_linear = np.array(history_times_linear)
history_plasma_descriptors_linear = np.array(history_plasma_descriptors_linear)



### Plotting
Plot a comparison of the super-fast no-GS mode versus a linear run that does solve the GS equation.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].plot(history_times, history_estimated_plasma_descriptors[:, 0], label="No GS")
ax[0].plot(history_times, history_plasma_descriptors_linear[:, 0], label="With GS (Linear)")
ax[0].set_xlabel("Time (s)")
ax[0].set_ylabel("ZCurrent")
ax[0].legend()

ax[1].plot(history_times, history_estimated_plasma_descriptors[:, 1], label="No GS")
ax[1].plot(history_times, history_plasma_descriptors_linear[:, 1], label="With GS (Linear)")
ax[1].set_xlabel("Time (s)")
ax[1].set_ylabel("Rx (m)")
ax[1].legend()

ax[2].plot(history_times, history_estimated_plasma_descriptors[:, 2], label="No GS")
ax[2].plot(history_times, history_plasma_descriptors_linear[:, 2], label="With GS (Linear)")
ax[2].set_xlabel("Time (s)")
ax[2].set_ylabel("Zx (m)")
ax[2].legend()

fig.tight_layout()